In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import talib
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:

stock_df = pd.read_csv('../data/processed/stock_combined.csv')
stock_df['Date'] = pd.to_datetime(stock_df['Date'])  # ensure datetime
stock_df = stock_df.sort_values(['Ticker', 'Date'])
print(stock_df.head())


In [ ]:
def calculate_indicators(df):
    df = df.copy()
    df = df.sort_values('Date')
    close = df['Close'].values
    high = df['High'].values
    low = df['Low'].values
    adj_close = df['Adj Close'].values if 'Adj Close' in df.columns else close
    
    df['SMA_20'] = talib.SMA(close, timeperiod=20)
    df['SMA_50'] = talib.SMA(close, timeperiod=50)
    df['SMA_200'] = talib.SMA(close, timeperiod=200)
    df['RSI_14'] = talib.RSI(close, timeperiod=14)
    df['MACD'], df['MACD_signal'], df['MACD_hist'] = talib.MACD(close)
    df['BB_upper'], df['BB_middle'], df['BB_lower'] = talib.BBANDS(close)
    df['ATR_14'] = talib.ATR(high, low, close, timeperiod=14)
    df['daily_return'] = (adj_close / pd.Series(adj_close).shift(1) - 1) * 100
    
    # Signals
    df['Golden_Cross'] = ((df['SMA_50'] > df['SMA_200']) & (df['SMA_50'].shift(1) <= df['SMA_200'].shift(1))).astype(int)
    df['Death_Cross'] = ((df['SMA_50'] < df['SMA_200']) & (df['SMA_50'].shift(1) >= df['SMA_200'].shift(1))).astype(int)
    df['RSI_Oversold'] = (df['RSI_14'] < 30).astype(int)
    df['RSI_Overbought'] = (df['RSI_14'] > 70).astype(int)
    df['MACD_Bullish'] = ((df['MACD'] > df['MACD_signal']) & (df['MACD'].shift(1) <= df['MACD_signal'].shift(1))).astype(int)
    df['MACD_Bearish'] = ((df['MACD'] < df['MACD_signal']) & (df['MACD'].shift(1) >= df['MACD_signal'].shift(1))).astype(int)
    return df

In [ ]:
all_tickers = []
for ticker in stock_df['Ticker'].unique():
    print(f"Processing {ticker}...")
    subset = stock_df[stock_df['Ticker'] == ticker]
    subset = calculate_indicators(subset)
    all_tickers.append(subset)

stock_indicators = pd.concat(all_tickers, ignore_index=True)
stock_indicators.to_parquet('../data/processed/stock_with_indicators.parquet')
print(f"Saved shape: {stock_indicators.shape}")

In [ ]:
ticker = 'AAPL'
df_plot = stock_indicators[stock_indicators['Ticker'] == ticker].copy()
df_plot.set_index('Date', inplace=True)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(df_plot.index, df_plot['Close'], label='Close', linewidth=1)
axes[0].plot(df_plot.index, df_plot['SMA_20'], label='SMA20', alpha=0.7)
axes[0].plot(df_plot.index, df_plot['SMA_50'], label='SMA50', alpha=0.7)
axes[0].plot(df_plot.index, df_plot['SMA_200'], label='SMA200', alpha=0.7)
axes[0].set_title(f'{ticker} Price with Moving Averages')
axes[0].legend()

axes[1].plot(df_plot.index, df_plot['RSI_14'], color='purple')
axes[1].axhline(70, color='red', linestyle='--', label='Overbought')
axes[1].axhline(30, color='green', linestyle='--', label='Oversold')
axes[1].fill_between(df_plot.index, 70, df_plot['RSI_14'], where=(df_plot['RSI_14'] > 70), color='red', alpha=0.3)
axes[1].fill_between(df_plot.index, 30, df_plot['RSI_14'], where=(df_plot['RSI_14'] < 30), color='green', alpha=0.3)
axes[1].set_title('RSI (14)')
axes[1].set_ylim(0, 100)
axes[1].legend()

axes[2].plot(df_plot.index, df_plot['MACD'], label='MACD', color='blue')
axes[2].plot(df_plot.index, df_plot['MACD_signal'], label='Signal', color='red')
colors = ['green' if v >= 0 else 'red' for v in df_plot['MACD_hist']]
axes[2].bar(df_plot.index, df_plot['MACD_hist'], color=colors, alpha=0.4, label='Histogram')
axes[2].axhline(0, color='black', alpha=0.5)
axes[2].set_title('MACD')
axes[2].legend()

plt.tight_layout()
plt.savefig(f'../reports/figures/{ticker}_indicators.png', dpi=150)
plt.show()

In [ ]:
signals = ['Golden_Cross', 'Death_Cross', 'RSI_Oversold', 'RSI_Overbought', 'MACD_Bullish']
summary = []
for ticker in stock_indicators['Ticker'].unique():
    df_t = stock_indicators[stock_indicators['Ticker'] == ticker]
    for sig in signals:
        entries = df_t[df_t[sig] == 1]
        if len(entries) > 0:
            avg_next = entries['daily_return'].shift(-1).mean()
            summary.append({'Ticker': ticker, 'Signal': sig, 'Count': len(entries), 'Avg_Next_Return_%': avg_next})
pd.DataFrame(summary).to_csv('../reports/signal_performance.csv', index=False)
print("Signal performance summary saved.")

In [ ]:
stock_indicators.to_csv('../data/processed/stock_with_indicators.csv', index=False)
print("Task 2 complete. Data saved to CSV.")